<a href="https://colab.research.google.com/github/Jayku88/22AIE301_Probabilistic_Reasoning/blob/main/22AIE301_Lab_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 22AIE301 - Probabilistic Reasoning
## Lab 07 - Belief Propagation

| | |
|---|---|
| **Name** | ADITHYAN ABHILASH SAJU |
| **Roll No** | AM.SC.U4AIE24006 |
| **Date** | 02-09-2026 |
| **Lab Slot** | 7 |

## Objectives

By the end of this lab you should be able to:

1. Recap **Variable Elimination** on the Student network (Lecture 8) for a roll-number-chosen query variable
2. Implement the **sum-product message** update $m_{i\to j}(x_j) = \sum_{x_i} \phi(x_i)\,\phi(x_i,x_j) \prod_{k \in \text{ne}(i)\setminus j} m_{k \to i}(x_i)$ from scratch
3. Run belief propagation by hand-tracing the **collect** and **distribute** passes on a 3-node chain, and verify every belief against brute-force enumeration
4. Generalize your chain code into a **readiness-driven message scheduler** that works on *any* tree (not just a chain), and apply it to a 5-node branching tree
5. Confirm the **2|E| message count** and cross-check every one of the 5 marginals against brute force
6. Apply the same generic BP function to a **star-shaped MRF**  and interpret which leaf dominates the result

Cells marked `___` need you to fill in code. Every part ends in an `assert` - if it raises, your
implementation (or your understanding of the formula) has a bug. Do **not** hardcode the expected
numbers from the lecture slides; your numbers are roll-parameterized and will differ.


---
## Part A - Variable Elimination (Student Network)

$D$ (Difficulty), $I$ (Intelligence), $G$ (Grade), $S$ (SAT), $L$ (Letter),
with $p(d,i,g,s,l) = p(d)\,p(i)\,p(g\mid i,d)\,p(s\mid i)\,p(l\mid g)$.

Your roll number selects **which single-variable marginal** you must compute - it will not always be
$L$ like the lecture example, so do not assume the lecture's `0.502` answer applies to you.


In [ ]:
import numpy as np
import itertools

ROLL = 6  # <-- enter your roll number as an integer, e.g. ROLL = 42
rng = np.random.default_rng(ROLL)

CARD = {'D': 2, 'I': 2, 'G': 3, 'S': 2, 'L': 2}
VARS5 = ['D', 'I', 'G', 'S', 'L']

query_var = VARS5[ROLL % 5]
print("Your query variable this lab:", query_var)


In [ ]:
class Factor:
    """A table over a tuple of discrete variables. values.shape must match
    the cardinalities of self.vars, in order."""
    def __init__(self, variables, values):
        self.vars = tuple(variables)
        self.values = np.array(values, dtype=float)
        assert self.values.shape == tuple(CARD[v] for v in self.vars), \
            f"shape mismatch for {self.vars}: got {self.values.shape}"

    def __repr__(self):
        return f"Factor{self.vars}"


def factor_product(f1, f2):
    """Return the factor product f1 * f2 (K&F Algorithm 9.1 style)."""
    all_vars = list(f1.vars) + [v for v in f2.vars if v not in f1.vars]
    shape = tuple(CARD[v] for v in all_vars)
    out = ___  # <-- allocate an output array of the right shape (np.zeros(shape))
    for ind in itertools.product(*[range(CARD[v]) for v in all_vars]):
        assign = dict(zip(all_vars, ind))
        i1 = tuple(assign[v] for v in f1.vars)
        i2 = tuple(assign[v] for v in f2.vars)
        out[ind] = ___  # <-- f1.values[i1] * f2.values[i2]
    return Factor(all_vars, out)


def factor_marginalize(f, var):
    """Sum out `var` from factor f."""
    axis = f.vars.index(var)
    new_vars = tuple(v for v in f.vars if v != var)
    new_values = ___  # <-- f.values.sum(axis=axis)
    return Factor(new_vars, new_values)


In [ ]:
def variable_elimination(factors, query_vars, elim_order):
    """Run VE, eliminating each variable in elim_order via gather->multiply->sum-out,
    then multiply what's left and marginalize out anything not in query_vars."""
    factors = list(factors)
    for var in elim_order:
        relevant = [f for f in factors if var in f.vars]
        factors = [f for f in factors if var not in f.vars]
        if relevant:
            prod = relevant[0]
            for f in relevant[1:]:
                prod = ___  # <-- multiply prod with f
            factors.append(___)  # <-- marginalize `var` out of prod
    result = factors[0]
    for f in factors[1:]:
        result = factor_product(result, f)
    for v in list(result.vars):
        if v not in query_vars:
            result = factor_marginalize(result, v)
    return result


In [ ]:
# Fixed K&F Student-network CPDs (same numbers as the Lecture 8 slides)
p_d = np.array([0.6, 0.4])
p_i = np.array([0.7, 0.3])
p_s_given_i = np.array([[0.95, 0.05], [0.20, 0.80]])          # rows i0,i1 ; cols s0,s1
p_l_given_g = np.array([[0.10, 0.90], [0.40, 0.60], [0.99, 0.01]])  # rows g1,g2,g3 ; cols l0,l1
p_g_given_i_d = np.array([
    [[0.30, 0.40, 0.30], [0.05, 0.25, 0.70]],   # i0: d0, d1
    [[0.90, 0.08, 0.02], [0.50, 0.30, 0.20]],   # i1: d0, d1
])

fD = Factor(['D'], p_d)
fI = Factor(['I'], p_i)
fG = Factor(['I', 'D', 'G'], p_g_given_i_d)
fS = Factor(['I', 'S'], p_s_given_i)
fL = Factor(['G', 'L'], p_l_given_g)

elim_order = [v for v in VARS5 if v != query_var]  # any order is correct, only efficiency differs
result = ___  # <-- call variable_elimination on [fD, fI, fG, fS, fL] for [query_var], elim_order

print(f"p({query_var}) =", result.values)
assert np.isclose(result.values.sum(), 1.0), "a marginal distribution must sum to 1"
print("Part A passed.")


---
## Part B - Belief Propagation on a 3-Node Chain

Chain example $X_1 - X_2 - X_3$,  with **your own** roll-parameterized
potentials. $\phi_2 \equiv \phi_3 \equiv 1$ (only $X_1$ carries a nontrivial unary potential, as in
the lecture), and $\phi_{12}, \phi_{23}$ are random $2\times2$ tables.


In [ ]:
phi1 = rng.integers(1, 6, size=2).astype(float)     # phi1(0), phi1(1)
phi2 = np.ones(2)
phi3 = np.ones(2)
phi12 = rng.integers(1, 4, size=(2, 2)).astype(float)  # phi12[x1, x2]
phi23 = rng.integers(1, 4, size=(2, 2)).astype(float)  # phi23[x2, x3]

print("phi1 =", phi1)
print("phi12 =\n", phi12)
print("phi23 =\n", phi23)


In [ ]:
def msg_1to2(phi1, phi12):
    """m_{1->2}(x2) = sum_{x1} phi1(x1) phi12(x1,x2)"""
    return np.array([___ for x2 in range(2)])  # <-- sum(phi1[x1]*phi12[x1,x2] for x1 in range(2))

def msg_3to2(phi3, phi23):
    """m_{3->2}(x2) = sum_{x3} phi3(x3) phi23(x2,x3)"""
    return np.array([___ for x2 in range(2)])  # <-- sum(phi3[x3]*phi23[x2,x3] for x3 in range(2))

m1to2 = msg_1to2(phi1, phi12)
m3to2 = msg_3to2(phi3, phi23)
print("m_1->2 =", m1to2)
print("m_3->2 =", m3to2)


In [ ]:
# Collect pass complete: X2 has heard from both its neighbours -> read off p(x2)
belief2 = phi2 * m1to2 * m3to2
belief2 = ___  # <-- normalize belief2 so it sums to 1
print("p(x2) =", belief2)


In [ ]:
# Distribute pass: X2 sends outward to X1 and X3, reusing m3to2 / m1to2 (no new info re-derived)
m2to1 = np.array([sum(phi12[x1, x2] * m3to2[x2] for x2 in range(2)) for x1 in range(2)])
belief1 = phi1 * m2to1
belief1 = ___  # <-- normalize

m2to3 = np.array([___ for x3 in range(2)])  # <-- sum(phi23[x2,x3]*m1to2[x2] for x2 in range(2))
belief3 = phi3 * m2to3
belief3 = ___  # <-- normalize

print("p(x1) =", belief1)
print("p(x3) =", belief3)


In [ ]:
# Ground truth: brute-force the full joint and marginalize directly.
joint = np.zeros((2, 2, 2))
for x1, x2, x3 in itertools.product(range(2), repeat=3):
    joint[x1, x2, x3] = phi1[x1] * phi2[x2] * phi3[x3] * phi12[x1, x2] * phi23[x2, x3]
Z = joint.sum()
p1_bf = joint.sum(axis=(1, 2)) / Z
p2_bf = joint.sum(axis=(0, 2)) / Z
p3_bf = joint.sum(axis=(0, 1)) / Z

assert np.allclose(belief1, p1_bf), (belief1, p1_bf)
assert np.allclose(belief2, p2_bf), (belief2, p2_bf)
assert np.allclose(belief3, p3_bf), (belief3, p3_bf)
print("Part B passed: chain BP beliefs match brute-force marginals.")


---
## Part C - A Generic Tree BP Engine (5-Node Branching Tree)

Now generalize Part B into code that works on **any tree**, not just a 3-node chain: a readiness
queue that fires a message $m_{i\to j}$ as soon as $i$ has heard from every neighbour except $j$, applied to the 5-node branching tree
$X_1 - X_2 - X_3$, $X_2 - X_4 - X_5$.


In [ ]:
neighbors = {1: [2], 2: [1, 3, 4], 3: [2], 4: [2, 5], 5: [4]}
edges = [(1, 2), (2, 3), (2, 4), (4, 5)]

phi = {i: rng.integers(1, 4, size=2).astype(float) for i in range(1, 6)}
phi_pair = {(a, b): rng.integers(1, 4, size=(2, 2)).astype(float) for (a, b) in edges}

def get_pair(i, j, phi_pair):
    """Return the pairwise table indexed as pair[x_i, x_j], regardless of storage direction."""
    if (i, j) in phi_pair:
        return phi_pair[(i, j)]
    else:
        return ___  # <-- phi_pair[(j, i)] needs transposing so axis 0 is x_i: phi_pair[(j,i)].T


In [ ]:
def compute_message(i, j, phi, phi_pair, messages, neighbors):
    """m_{i->j}(x_j) = sum_{x_i} phi_i(x_i) * phi_ij(x_i,x_j) * prod_{k in ne(i)\\{j}} m_{k->i}(x_i)"""
    prod_incoming = np.ones(2)
    for k in neighbors[i]:
        if k != j:
            prod_incoming = ___  # <-- multiply prod_incoming by messages[(k, i)]
    pair = get_pair(i, j, phi_pair)
    msg = np.zeros(2)
    for xj in range(2):
        msg[xj] = ___  # <-- sum(phi[i][xi]*prod_incoming[xi]*pair[xi,xj] for xi in range(2))
    return msg


In [ ]:
def run_belief_propagation(phi, phi_pair, neighbors, edges):
    """Readiness-driven scheduler: repeatedly fire any message whose sender has already
    heard from all neighbours except the recipient, until every directed edge has fired."""
    messages = {}
    sent = set()
    all_dir_edges = [(i, j) for (a, b) in edges for (i, j) in [(a, b), (b, a)]]

    while len(sent) < len(all_dir_edges):
        progressed = False
        for (i, j) in all_dir_edges:
            if (i, j) in sent:
                continue
            others = [k for k in neighbors[i] if k != j]
            if ___:  # <-- condition: has i heard from every node in `others`? (all (k,i) in sent)
                messages[(i, j)] = compute_message(i, j, phi, phi_pair, messages, neighbors)
                sent.add((i, j))
                progressed = True
        assert progressed, "deadlock: no node was ready to send -- check your readiness condition"

    beliefs = {}
    for i in neighbors:
        b = phi[i].copy()
        for k in neighbors[i]:
            b = b * messages[(k, i)]
        beliefs[i] = b / b.sum()

    return beliefs, messages, sent

beliefs5, messages5, sent5 = run_belief_propagation(phi, phi_pair, neighbors, edges)

assert len(sent5) == 2 * len(edges), "should be exactly 2|E| messages"
print(f"Total messages sent: {len(sent5)} (2|E| = {2*len(edges)})")
for i in range(1, 6):
    print(f"p(x{i}) =", beliefs5[i])


In [ ]:
# Ground truth via brute force over all 2^5 assignments.
joint5 = np.zeros((2,) * 5)
for assign in itertools.product(range(2), repeat=5):
    a = dict(zip(range(1, 6), assign))
    val = 1.0
    for i in range(1, 6):
        val *= phi[i][a[i]]
    for (u, v) in edges:
        val *= phi_pair[(u, v)][a[u], a[v]]
    joint5[assign] = val
Z5 = joint5.sum()

for i in range(1, 6):
    axes = tuple(k for k in range(5) if k != i - 1)
    marg = joint5.sum(axis=axes) / Z5
    assert np.allclose(marg, beliefs5[i]), (i, marg, beliefs5[i])

print("Part C passed: all 5 tree marginals match brute-force enumeration.")


---
## Part D - Why Bother? Counting the Work

Compare the cost of getting **all five** marginals of the tree in Part C via BP, versus getting them
via five independent VE runs.


In [ ]:
# BP: one collect + one distribute pass = 2|E| messages total, however many marginals you want.
bp_message_count = len(sent5)

# Naive VE: a fresh elimination run per query variable. Each run eliminates the 4 non-query
# variables of the tree, and (ignoring exact factor sizes) that's roughly len(edges) "eliminate
# a variable" steps per run.
num_queries = 5
naive_ve_steps = num_queries * len(edges)

print(f"BP total messages (all 5 marginals): {bp_message_count}")
print(f"Naive VE 'eliminate a variable' steps (all 5 marginals, one VE run each): {naive_ve_steps}")
assert bp_message_count < naive_ve_steps
print("BP amortizes the shared computation across every query -- this is the whole point of Lecture 9.")


**Short answer (fill in):** In your own words, why does the number of BP messages stay fixed at
$2|E|$ no matter how many of the five marginals you need, while the naive repeated-VE approach's
cost grows with the number of queries?

*Your answer:* ___


---
## Part E - Star Network

A central node $Y$ (node `0`) connects to three leaves $X_1, X_2, X_3$ (nodes `1,2,3`), no edges
among the leaves. $\phi_Y \equiv 1$. Leaf unary potentials and the (roll-parameterized) "agree/
disagree" pairwise potential are generated below. Reuse your `compute_message` /
`run_belief_propagation` from Part C - a star is just another tree.


In [ ]:
star_neighbors = {0: [1, 2, 3], 1: [0], 2: [0], 3: [0]}
star_edges = [(0, 1), (0, 2), (0, 3)]

phi_star = {0: np.ones(2)}
for leaf in [1, 2, 3]:
    phi_star[leaf] = rng.integers(1, 6, size=2).astype(float)

phi_pair_star = {}
for (a, b) in star_edges:
    w = rng.integers(2, 5)  # "agree" weight; "disagree" weight fixed at 1
    phi_pair_star[(a, b)] = np.array([[float(w), 1.0], [1.0, float(w)]])

for leaf in [1, 2, 3]:
    print(f"phi_{leaf} =", phi_star[leaf], " | agree-weight on edge (0,{}) =".format(leaf), phi_pair_star[(0, leaf)][0, 0])


In [ ]:
beliefs_star, messages_star, sent_star = ___  # <-- call run_belief_propagation on the star's phi_star, phi_pair_star, star_neighbors, star_edges

print("p(Y) =", beliefs_star[0])
assert len(sent_star) == 2 * len(star_edges)


In [ ]:
joint_star = np.zeros((2, 2, 2, 2))
for assign in itertools.product(range(2), repeat=4):
    a = dict(zip([0, 1, 2, 3], assign))
    val = 1.0
    for i in [0, 1, 2, 3]:
        val *= phi_star[i][a[i]]
    for (u, v) in star_edges:
        val *= phi_pair_star[(u, v)][a[u], a[v]]
    joint_star[assign] = val
Z_star = joint_star.sum()
p0_bf = joint_star.sum(axis=(1, 2, 3)) / Z_star

assert np.allclose(beliefs_star[0], p0_bf), (beliefs_star[0], p0_bf)
print("Part E passed: star belief matches brute force.")


**Short answer (fill in):** Looking at your own `phi_star` values and the messages each leaf sent
to $Y$ (print `messages_star[(1,0)]`, `messages_star[(2,0)]`, `messages_star[(3,0)]` if you like) —
which leaf's potential most influenced $p(Y)$, and why does a leaf with a more *symmetric* unary
potential send a flatter message?

*Your answer:* ___


---
## Submission Checklist
-  Your **Roll No** is set correctly in the Setup cell (`ROLL`).
-  Save the notebook: **File → Save** (or Ctrl+S).

**Rename the file as:** `Lab07_<YourRollNo>.ipynb` and convert to PDF before submitting.